<a href="https://colab.research.google.com/github/sanmquin/AI/blob/main/Grafiko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install pymongo if not already present
!pip install pymongo dnspython

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm a successful connection
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"An error occurred: {e}")
    print("\nMake sure you have added 'MONGODB_URI' to your Colab Secrets (the key icon on the left).")

Pinged your deployment. You successfully connected to MongoDB!


In [3]:
# Access Finder DB collections
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']

# 1) Get the latest version in ChannelDescriptions_clusters
latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None

print(f"Latest cluster version: {latest_version}")



Latest cluster version: 2


In [8]:
# 2) Get the _id of the business cluster in the latest version
if latest_version is None:
    print('No clusters found.')
    business_cluster = None
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })

    if business_cluster:
        business_cluster_id = business_cluster['_id']
        print(f"Business cluster id: {business_cluster_id}")
        print(f"Business cluster name: {business_cluster.get('name')}")
    else:
        business_cluster_id = None
        print(f"No business cluster found in version {latest_version}.")



Business cluster id: 69e41878685f30ed081ec5a6
Business cluster name: Business, Venture Capital, and Entrepreneurship


In [9]:
# 3) Get channels in the business cluster (id + title)
if business_cluster_id is None:
    print('Cannot fetch channels without a business cluster id.')
else:
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs if d.get('textId')]

    channel_docs = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1}
    ))

    channel_by_id = {c['channelId']: c for c in channel_docs}

    print(f"Channels in business cluster ({len(channel_ids)} total):")
    for cid in channel_ids:
        c = channel_by_id.get(cid)
        title = c.get('title') if c else None
        print({'id': cid, 'title': title})



Channels in business cluster (27 total):
{'id': 'UC9cn0TuPq4dnbTY-CBsm8XA', 'title': 'a16z'}
{'id': 'UCESLZhusAkFfsNsApnjF_Cg', 'title': 'All-In Podcast'}
{'id': 'UCcefcZRL2oaA_uBNeo5UOWg', 'title': 'Y Combinator'}
{'id': 'UCvxm0qTrGN_1LMYgUaftWyQ', 'title': 'Peter H. Diamandis'}
{'id': 'UCPjNBjflYl0-HQtUvOx0Ibw', 'title': 'Greg Isenberg'}
{'id': 'UC-yRDvpR99LUc5l7i7jLzew', 'title': 'Bg2 Pod'}
{'id': 'UCJEDniyP_YtcsXikBELqicw', 'title': None}
{'id': 'UCBH5VZE_Y4F3CMcPIzPEB5A', 'title': None}
{'id': 'UC1LpsuAUaKoMzzJSEt5WImw', 'title': 'Asianometry'}
{'id': 'UCznv7Vf9nBdJYvBagFdAHWw', 'title': 'Tim Ferriss'}
{'id': 'UCK-zlnUfoDHzUwXcbddtnkg', 'title': None}
{'id': 'UC6t1O76G0jYXOAoYCm153dA', 'title': "Lenny's Podcast"}
{'id': 'UCIHdDJ0tjn_3j-FS7s_X1kQ', 'title': 'Valuetainment'}
{'id': 'UCf0PBRjhf0rF8fWBIxTuoWA', 'title': '20VC with Harry Stebbings'}
{'id': 'UCWrF0oN6unbXrWsTN7RctTw', 'title': 'Sequoia Capital'}
{'id': 'UCyaN6mg5u8Cjy2ZI4ikWaug', 'title': None}
{'id': 'UCqvaXJ1K3HheTPNj

In [6]:
# Optional: inspect the business cluster document
if business_cluster_id is not None:
    import pprint
    pprint.pprint(business_cluster)


## Subscriptions analysis for business-cluster channels
Counts subscriptions per channel and builds an owner-overlap matrix for channel pairs.


In [ ]:
# 4) Query subscriptions for all business-cluster channels and count rows per channel
from collections import defaultdict

subscriptions_col = db['subscriptions']

if business_cluster_id is None:
    print('Cannot analyze subscriptions without a business cluster id.')
else:
    # Keep only unique channel ids while preserving order
    channel_ids_unique = list(dict.fromkeys(channel_ids))

    # Pull relevant subscription rows once, then aggregate in Python
    subs_docs = list(subscriptions_col.find(
        {'channelId': {'$in': channel_ids_unique}},
        {
            '_id': 0,
            'channelId': 1,
            'authorId': 1,
            'ownerId': 1,
            'userId': 1,
            'createdBy': 1,
            'accountId': 1,
        }
    ))

    # Find which field contains the subscription owner / author id
    owner_candidates = ['authorId', 'ownerId', 'userId', 'createdBy', 'accountId']
    owner_field = None
    for field in owner_candidates:
        if any(doc.get(field) is not None for doc in subs_docs):
            owner_field = field
            break

    if owner_field is None:
        raise ValueError(
            'Could not infer owner field in subscriptions. '
            f'Checked: {owner_candidates}'
        )

    print(f"Owner field used for author/subscription owner: {owner_field}")

    # 1+2) Count subscriptions per channel (replicated for all channels)
    subscriptions_per_channel = defaultdict(int)
    owners_per_channel = defaultdict(set)

    for doc in subs_docs:
        cid = doc.get('channelId')
        owner = doc.get(owner_field)
        if cid is None:
            continue
        subscriptions_per_channel[cid] += 1
        if owner is not None:
            owners_per_channel[cid].add(owner)

    print('
Subscription count by channel id:')
    for cid in channel_ids_unique:
        title = channel_by_id.get(cid, {}).get('title')
        print(f"- {cid} | {title} => {subscriptions_per_channel[cid]} subscriptions")


In [ ]:
# 3+4+5) Build pairwise owner overlap matrix, print with labels, and save for reuse in Colab
import pandas as pd
from pathlib import Path

if business_cluster_id is None:
    print('Skipping matrix creation (no business cluster id).')
else:
    labels = []
    for cid in channel_ids_unique:
        title = channel_by_id.get(cid, {}).get('title') or 'Unknown title'
        labels.append(f"{title} ({cid})")

    # Matrix[i][j] = number of unique owners subscribed to both channels i and j
    matrix = []
    for cid_i in channel_ids_unique:
        row = []
        owners_i = owners_per_channel.get(cid_i, set())
        for cid_j in channel_ids_unique:
            owners_j = owners_per_channel.get(cid_j, set())
            row.append(len(owners_i & owners_j))
        matrix.append(row)

    overlap_df = pd.DataFrame(matrix, index=labels, columns=labels)

    print('Author overlap matrix (pairwise shared subscription owners):')
    print(overlap_df)

    # Save in /content so it persists for the Colab session and is easy to download/reload
    out_dir = Path('/content/business_cluster_subscriptions')
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / 'owner_overlap_matrix.csv'
    pkl_path = out_dir / 'owner_overlap_matrix.pkl'
    counts_path = out_dir / 'subscriptions_per_channel.csv'

    overlap_df.to_csv(csv_path)
    overlap_df.to_pickle(pkl_path)

    counts_df = pd.DataFrame([
        {
            'channelId': cid,
            'title': channel_by_id.get(cid, {}).get('title'),
            'subscriptions': subscriptions_per_channel.get(cid, 0),
            'unique_owners': len(owners_per_channel.get(cid, set()))
        }
        for cid in channel_ids_unique
    ])
    counts_df.to_csv(counts_path, index=False)

    print(f"
Saved matrix CSV: {csv_path}")
    print(f"Saved matrix PKL: {pkl_path}")
    print(f"Saved channel counts CSV: {counts_path}")
